# Student Workbook: Random Forest

Same class, same charts, same order -- but the core calculations are left for you to fill in. Nothing here is
graded.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

pd.set_option("display.precision", 3)

Last session ended with an unconstrained decision tree that scored a perfect 1.000 on training data -- and clearly worse on data it hadn't seen. Let's rebuild that exact tree and look at its region again.

In [2]:
pass_df = pd.read_csv("../data/exam_pass_3d.csv")
X = pass_df[["hours_studied", "practice_problems"]]
y = pass_df["passed"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
pass_df.head()

,hours_studied,practice_problems,sleep_hours,passed
0,7.0,9,8.5,1
1,6.6,18,6.0,0
2,0.7,1,8.1,0
3,7.0,16,4.8,0
4,3.2,20,7.2,0


### ✏️ Try it yourself

Fit a `DecisionTreeClassifier(max_depth=None)` on `X_train, y_train`, and print its train and test accuracy.

In [3]:
# TODO: fit deep_tree_model and print its train/test accuracy

<details>
<summary>Show solution</summary>

```python
deep_tree_model = DecisionTreeClassifier(max_depth=None, random_state=0)
deep_tree_model.fit(X_train, y_train)
print(f"Train accuracy: {deep_tree_model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {deep_tree_model.score(X_test, y_test):.3f}")
```

</details>

Continuing with the worked model:

In [4]:
deep_tree_model = DecisionTreeClassifier(max_depth=None, random_state=0)
deep_tree_model.fit(X_train, y_train)
print(f"Train accuracy: {deep_tree_model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {deep_tree_model.score(X_test, y_test):.3f}")

Train accuracy: 1.000
Test accuracy:  0.879


<details>
<summary>Show code</summary>

```python
grid_hours = np.linspace(X["hours_studied"].min(), X["hours_studied"].max(), 200)
grid_practice = np.linspace(X["practice_problems"].min(), X["practice_problems"].max(), 200)
grid_x, grid_y = np.meshgrid(grid_hours, grid_practice)
grid_points = pd.DataFrame({"hours_studied": grid_x.ravel(), "practice_problems": grid_y.ravel()})

deep_tree_grid = deep_tree_model.predict(grid_points).reshape(grid_x.shape)

region_figure = go.Figure()
region_figure.add_trace(
    go.Contour(
        x=grid_hours, y=grid_practice, z=deep_tree_grid,
        showscale=False, colorscale=[[0, "rgba(220,38,38,0.15)"], [1, "rgba(22,163,74,0.15)"]],
        contours=dict(start=0, end=1, size=1, coloring="fill"),
        line=dict(width=0), hoverinfo="skip",
    )
)
for passed_value, color, label in [(0, "#dc2626", "actual: failed"), (1, "#16a34a", "actual: passed")]:
    subset = pass_df[pass_df["passed"] == passed_value]
    region_figure.add_trace(
        go.Scatter(
            x=subset["hours_studied"], y=subset["practice_problems"], mode="markers",
            marker=dict(size=9, color=color, line=dict(width=1, color="white")), name=label,
        )
    )
region_figure.update_layout(
    title="Last Session's Unconstrained Tree (jagged, overfit)",
    xaxis_title="Hours studied", yaxis_title="Practice problems",
    template="plotly_white", width=750, height=550,
)
region_figure.show()
```

</details>

In [5]:
grid_hours = np.linspace(X["hours_studied"].min(), X["hours_studied"].max(), 200)
grid_practice = np.linspace(X["practice_problems"].min(), X["practice_problems"].max(), 200)
grid_x, grid_y = np.meshgrid(grid_hours, grid_practice)
grid_points = pd.DataFrame({"hours_studied": grid_x.ravel(), "practice_problems": grid_y.ravel()})

deep_tree_grid = deep_tree_model.predict(grid_points).reshape(grid_x.shape)

region_figure = go.Figure()
region_figure.add_trace(
    go.Contour(
        x=grid_hours, y=grid_practice, z=deep_tree_grid,
        showscale=False, colorscale=[[0, "rgba(220,38,38,0.15)"], [1, "rgba(22,163,74,0.15)"]],
        contours=dict(start=0, end=1, size=1, coloring="fill"),
        line=dict(width=0), hoverinfo="skip",
    )
)
for passed_value, color, label in [(0, "#dc2626", "actual: failed"), (1, "#16a34a", "actual: passed")]:
    subset = pass_df[pass_df["passed"] == passed_value]
    region_figure.add_trace(
        go.Scatter(
            x=subset["hours_studied"], y=subset["practice_problems"], mode="markers",
            marker=dict(size=9, color=color, line=dict(width=1, color="white")), name=label,
        )
    )
region_figure.update_layout(
    title="Last Session's Unconstrained Tree (jagged, overfit)",
    xaxis_title="Hours studied", yaxis_title="Practice problems",
    template="plotly_white", width=750, height=550,
)
region_figure.show()

## The Core Idea: Many Slightly-Different Trees

Imagine instead of asking one tree, we trained 100 trees, each on its own randomly-drawn version of the same
class roster -- some students appear twice, some don't appear at all -- and then let them vote on every
prediction. This "randomly-drawn version" is called a **bootstrap sample**: drawing rows with replacement from
the original data, up to the original size.

> ### 💭 Think about it
>
> If 100 people each guess a number independently and you average their guesses, why does the average often land closer to the truth than any individual guess?

## Bootstrap Sampling, Visualized

### ✏️ Try it yourself

Using `rng.choice(sample_slice.index, size=len(sample_slice), replace=True)`, draw 3 bootstrap resamples of `sample_slice` and print each one's sorted indices.

In [6]:
rng = np.random.default_rng(seed=0)
sample_slice = pass_df.iloc[:8].reset_index(drop=True)
print("Original roster indices:", list(sample_slice.index))

# TODO: draw 3 bootstrap resamples and print each one's sorted indices

Original roster indices: [0, 1, 2, 3, 4, 5, 6, 7]


<details>
<summary>Show solution</summary>

```python
rng = np.random.default_rng(seed=0)
sample_slice = pass_df.iloc[:8].reset_index(drop=True)
print("Original roster indices:", list(sample_slice.index))

for resample_number in range(1, 4):
    resample_indices = rng.choice(sample_slice.index, size=len(sample_slice), replace=True)
    print(f"Bootstrap resample {resample_number}:", sorted(resample_indices))
```

</details>

Notice some indices repeat within a resample, and some are missing entirely -- each tree in the forest will be trained on a roster that looks a little different from the original, and different from every other tree's roster.

## Fitting a Random Forest

### ✏️ Try it yourself

Fit a `RandomForestClassifier(n_estimators=100)` on `X_train, y_train`, and print its train and test accuracy.

In [7]:
# TODO: fit forest_model and print its train/test accuracy

<details>
<summary>Show solution</summary>

```python
forest_model = RandomForestClassifier(n_estimators=100, random_state=0)
forest_model.fit(X_train, y_train)
print(f"Train accuracy: {forest_model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {forest_model.score(X_test, y_test):.3f}")
```

</details>

Continuing with the worked model:

In [8]:
forest_model = RandomForestClassifier(n_estimators=100, random_state=0)
forest_model.fit(X_train, y_train)
print(f"Train accuracy: {forest_model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {forest_model.score(X_test, y_test):.3f}")

forest_grid = forest_model.predict(grid_points).reshape(grid_x.shape)

forest_region_figure = go.Figure()
forest_region_figure.add_trace(
    go.Contour(
        x=grid_hours, y=grid_practice, z=forest_grid,
        showscale=False, colorscale=[[0, "rgba(220,38,38,0.15)"], [1, "rgba(22,163,74,0.15)"]],
        contours=dict(start=0, end=1, size=1, coloring="fill"),
        line=dict(width=0), hoverinfo="skip",
    )
)
for passed_value, color, label in [(0, "#dc2626", "actual: failed"), (1, "#16a34a", "actual: passed")]:
    subset = pass_df[pass_df["passed"] == passed_value]
    forest_region_figure.add_trace(
        go.Scatter(
            x=subset["hours_studied"], y=subset["practice_problems"], mode="markers",
            marker=dict(size=9, color=color, line=dict(width=1, color="white")), name=label,
        )
    )
forest_region_figure.update_layout(
    title="Random Forest Region (100 trees, averaged)",
    xaxis_title="Hours studied", yaxis_title="Practice problems",
    template="plotly_white", width=750, height=550,
)
forest_region_figure.show()

Train accuracy: 1.000
Test accuracy:  0.939


Same training accuracy as the single deep tree -- but noticeably better on data it never saw, and the region itself looks smoother, less jagged.

## Does the Forest Really Overfit Less?

<details>
<summary>Show code</summary>

```python
comparison_table = pd.DataFrame([
    {
        "model": "Single deep tree (max_depth=None)",
        "train_accuracy": deep_tree_model.score(X_train, y_train),
        "test_accuracy": deep_tree_model.score(X_test, y_test),
    },
    {
        "model": "Random forest (100 trees)",
        "train_accuracy": forest_model.score(X_train, y_train),
        "test_accuracy": forest_model.score(X_test, y_test),
    },
])
comparison_table
```

</details>

In [9]:
comparison_table = pd.DataFrame([
    {
        "model": "Single deep tree (max_depth=None)",
        "train_accuracy": deep_tree_model.score(X_train, y_train),
        "test_accuracy": deep_tree_model.score(X_test, y_test),
    },
    {
        "model": "Random forest (100 trees)",
        "train_accuracy": forest_model.score(X_train, y_train),
        "test_accuracy": forest_model.score(X_test, y_test),
    },
])
comparison_table

,model,train_accuracy,test_accuracy
0,Single deep tree (max_depth=None),1.0,0.879
1,Random forest (100 trees),1.0,0.939


> ### 💭 Think about it
>
> The forest's training accuracy isn't lower than the single tree's -- so why is its test accuracy higher?

## How Many Trees Do We Need?

### ✏️ Try it yourself

For `n_estimators` in `[1, 5, 50]`: fit a `RandomForestClassifier`, and record its test accuracy. Compare the three numbers -- and separately, compare how smooth each one's boundary region looks if you re-plot it.

In [10]:
for n in [1, 5, 50]:
    # TODO: fit a RandomForestClassifier(n_estimators=n), then print n and its test accuracy
    pass

<details>
<summary>Show solution</summary>

```python
for n in [1, 5, 50]:
    sweep_model = RandomForestClassifier(n_estimators=n, random_state=0)
    sweep_model.fit(X_train, y_train)
    print(f"n_estimators={n}: test accuracy = {sweep_model.score(X_test, y_test):.3f}")
```

</details>

Continuing with the full sweep so we can plot it:

<details>
<summary>Show code</summary>

```python
tree_counts = [1, 2, 5, 10, 25, 50, 100, 200]
sweep_accuracies = []
for n in tree_counts:
    sweep_model = RandomForestClassifier(n_estimators=n, random_state=0)
    sweep_model.fit(X_train, y_train)
    sweep_accuracies.append(sweep_model.score(X_test, y_test))

sweep_figure = go.Figure()
sweep_figure.add_trace(
    go.Scatter(x=tree_counts, y=sweep_accuracies, mode="lines+markers",
               line=dict(color="#16a34a", width=3), marker=dict(size=9))
)
sweep_figure.update_layout(
    title="Test Accuracy vs. Number of Trees",
    xaxis_title="n_estimators", yaxis_title="test accuracy",
    template="plotly_white", width=700, height=450,
)
sweep_figure.show()
```

</details>

In [11]:
tree_counts = [1, 2, 5, 10, 25, 50, 100, 200]
sweep_accuracies = []
for n in tree_counts:
    sweep_model = RandomForestClassifier(n_estimators=n, random_state=0)
    sweep_model.fit(X_train, y_train)
    sweep_accuracies.append(sweep_model.score(X_test, y_test))

sweep_figure = go.Figure()
sweep_figure.add_trace(
    go.Scatter(x=tree_counts, y=sweep_accuracies, mode="lines+markers",
               line=dict(color="#16a34a", width=3), marker=dict(size=9))
)
sweep_figure.update_layout(
    title="Test Accuracy vs. Number of Trees",
    xaxis_title="n_estimators", yaxis_title="test accuracy",
    template="plotly_white", width=700, height=450,
)
sweep_figure.show()

> ### 🧑‍🏫 Instructor note
>
> Contrast with last session: more DEPTH could actively hurt test accuracy. More TREES essentially never hurts
> accuracy here -- it just costs more compute past a certain point. That's a genuinely different kind of knob.

## Feature Importance, Averaged Across the Forest

Back to all three features, including the noise column `sleep_hours`.

### ✏️ Try it yourself

Fit a single `DecisionTreeClassifier(max_depth=3)` and a `RandomForestClassifier(n_estimators=100)` on all three features, and make a grouped bar chart comparing their `.feature_importances_`.

In [12]:
X3 = pass_df[["hours_studied", "practice_problems", "sleep_hours"]]
y3 = pass_df["passed"]

# TODO: fit single_tree_importance and forest_importance, then build a grouped go.Bar chart comparing them

<details>
<summary>Show solution</summary>

```python
X3 = pass_df[["hours_studied", "practice_problems", "sleep_hours"]]
y3 = pass_df["passed"]

single_tree_importance = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X3, y3).feature_importances_
forest_importance = RandomForestClassifier(n_estimators=100, random_state=0).fit(X3, y3).feature_importances_

importance_figure = go.Figure()
importance_figure.add_trace(go.Bar(x=list(X3.columns), y=single_tree_importance, name="single tree", marker=dict(color="#dc2626")))
importance_figure.add_trace(go.Bar(x=list(X3.columns), y=forest_importance, name="random forest", marker=dict(color="#16a34a")))
importance_figure.update_layout(
    barmode="group", title="Feature Importance: Single Tree vs. Random Forest",
    xaxis_title="feature", yaxis_title="importance",
    template="plotly_white", width=700, height=450,
)
importance_figure.show()
```

</details>

The single tree should give `sleep_hours` essentially zero importance. The forest gives it a small but nonzero score -- each split inside each tree only considers a random subset of features, so occasionally a tree is forced to split on `sleep_hours` simply because the useful features weren't in its sampled subset for that split.

## What We Covered Today

- A random forest trains many trees, each on a bootstrap resample of the data (and a random feature subset per split).
- Averaging/voting across trees smooths out any one tree's overfitting.
- More trees (`n_estimators`) essentially never hurts test accuracy, unlike more depth on a single tree.
- Feature importance from a forest is noisier but more informative than from a single tree -- a genuinely
  useless feature still gets a small, nonzero score.

## Final Check

1. Why does averaging many bootstrap-trained trees tend to overfit less than one deep tree?
2. Session 4 said more depth can hurt test accuracy. Does more `n_estimators` carry the same risk? Why or why not?
3. A colleague says "our forest ranks `sleep_hours` as more important than the single tree does, so the forest must know something the tree doesn't." What's wrong with that reasoning?

<details>
<summary>Show solution</summary>

1. Any single tree, given enough depth, can memorize noise specific to its training rows. Since each tree in
   the forest trains on a different bootstrap sample, their individual mistakes tend to be different from each
   other -- averaging cancels many of them out.
2. No, not the same risk. Adding more trees just adds more independent "opinions" to average over; it doesn't
   give any single tree more room to memorize noise. Compute cost is the main downside of a large `n_estimators`.
3. The forest's nonzero score for a noise feature isn't a sign the forest found real signal -- it's a side
   effect of each split only considering a random subset of features, so the noise feature occasionally gets
   picked "by default." The single tree's zero score is actually the more informative signal here.

</details>